In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    average_precision_score
)
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_predict
import joblib
import json


In [2]:
df = pd.read_csv(Path.cwd().parent.parent/ "data"/ "processed"/ "heart_disease_health_indicators_BRFSS2015.csv")
X = df.drop(columns=["HeartDiseaseorAttack"])
y = df["HeartDiseaseorAttack"]
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X shape: (295751, 21)
y shape: (295751,)

Target distribution:
HeartDiseaseorAttack
0.0    270728
1.0     25023
Name: count, dtype: int64
X_train: (236600, 21)
X_test : (59151, 21)
y_train: (236600,)
y_test : (59151,)


# Test Models to Generalize Params:

### Model A:
- n_estimators=300
- max_depth=4
- learning_rate=0.05


In [38]:
model_a = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
model_a.fit(X_train, y_train)

y_pred_a = model_a.predict(X_test)
y_prob_a = model_a.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, y_pred_a)

print("Accuracy :", accuracy_score(y_test, y_pred_a))
print("Precision:", precision_score(y_test, y_pred_a))
print("Recall   :", recall_score(y_test, y_pred_a))
print("F1       :", f1_score(y_test, y_pred_a))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_a))
print(cm)

#test thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print("\nTHRESHOLD RANGE TEST:")
for threshold in thresholds:
    y_pred_custom = (y_prob_a >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_custom)
    recall = recall_score(y_test, y_pred_custom)
    f1 = f1_score(y_test, y_pred_custom)

    print(
        f"Threshold={threshold:.1f} | "
        f"Precision={precision:.3f} | "
        f"Recall={recall:.3f} | "
        f"F1={f1:.3f}"
    )

print("Min :", y_prob_a.min())
print("Max :", y_prob_a.max())
print("Mean:", y_prob_a.mean())

Accuracy : 0.9177359638890298
Precision: 0.5807200929152149
Recall   : 0.0999000999000999
F1       : 0.17047391749062393
ROC-AUC  : 0.8614019286959117
[[53785   361]
 [ 4505   500]]

THRESHOLD RANGE TEST:
Threshold=0.1 | Precision=0.247 | Recall=0.782 | F1=0.375
Threshold=0.2 | Precision=0.338 | Recall=0.559 | F1=0.421
Threshold=0.3 | Precision=0.415 | Recall=0.368 | F1=0.390
Threshold=0.4 | Precision=0.503 | Recall=0.207 | F1=0.293
Threshold=0.5 | Precision=0.581 | Recall=0.100 | F1=0.170
Threshold=0.6 | Precision=0.688 | Recall=0.049 | F1=0.091
Threshold=0.7 | Precision=0.694 | Recall=0.012 | F1=0.023
Threshold=0.8 | Precision=0.750 | Recall=0.001 | F1=0.001
Threshold=0.9 | Precision=0.000 | Recall=0.000 | F1=0.000
Min : 0.0007556155
Max : 0.84443945
Mean: 0.084161274


c:\Users\giaba\anaconda3\envs\vis_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Model B:
- n_estimators=300
- max_depth=4
- learning_rate=0.1

In [ ]:
model_b = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)

model_b.fit(X_train, y_train)

y_pred_b = model_b.predict(X_test)
y_prob_b = model_b.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_b))
print("Precision:", precision_score(y_test, y_pred_b))
print("Recall   :", recall_score(y_test, y_pred_b))
print("F1       :", f1_score(y_test, y_pred_b))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_b))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_b))

threshold = 0.2

y_pred_b_02 = (y_prob_b >= threshold).astype(int)
print("\nTHRESHOLD TEST 0.2:")
print("Precision:", precision_score(y_test, y_pred_b_02))
print("Recall   :", recall_score(y_test, y_pred_b_02))
print("F1       :", f1_score(y_test, y_pred_b_02))

Accuracy : 0.9172964108806275
Precision: 0.5621562156215621
Recall   : 0.1020979020979021
F1       : 0.1728102806898884
ROC-AUC  : 0.8606351337134774

Confusion Matrix:
[[53748   398]
 [ 4494   511]]

 THRESHOLD TEST 0.2:
Precision: 0.3363537513591881
Recall   : 0.5562437562437562
F1       : 0.4192139737991266


### Model C:
- n_estimators=600
- max_depth=4
- learning_rate=0.05

In [40]:
model_c = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

model_c.fit(X_train, y_train)

y_prob_c = model_c.predict_proba(X_test)[:, 1]
y_pred_c = (y_prob_c >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred_c))
print("Precision:", precision_score(y_test, y_pred_c))
print("Recall   :", recall_score(y_test, y_pred_c))
print("F1       :", f1_score(y_test, y_pred_c))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_c))

Accuracy : 0.9174316579601359
Precision: 0.566410537870472
Recall   : 0.1030969030969031
F1       : 0.1744421906693712
ROC-AUC  : 0.8608320372421137


# Grid Search:

### Hyperparameter Finetuning

In [24]:
# ============================================================
# 1. Base model
# ============================================================

model = XGBClassifier(
    max_depth = 3,
    subsample = 0.9,
    colsample_bytree = 0.8,
    min_child_weight = 400,
    gamma = 0.10,
    reg_alpha = 0,
    reg_lambda = 2,
    eval_metric="logloss"
)

# ============================================================
# 2. Hyperparameter cần fine-tune
# ============================================================

param_grid = {
    "learning_rate": [0.08, 0.10, 0.12, 0.15, 0.18, 0.20],
    "n_estimators": [200, 250, 300, 350, 400, 450]
}

# ============================================================
# 3. GridSearchCV
# ============================================================

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

# ============================================================
# 4. Best model
# ============================================================

best_model = grid_search.best_estimator_

print("\n===== BEST MODEL =====")
print(grid_search.best_params_)
print(f"Best CV PR-AUC: {grid_search.best_score_:.6f}")


# ============================================================
# 5. Compare all configurations
# ============================================================

results = pd.DataFrame(grid_search.cv_results_)

results = results.drop(
    columns=[
        "mean_fit_time",
        "std_fit_time",
        "mean_score_time",
        "std_score_time",
        "params",
        "split0_test_score",
        "split1_test_score",
        "split2_test_score",
        "split0_train_score",
        "split1_train_score",
        "split2_train_score"
        ]
)

results["gap"] = (
    results["mean_train_score"]
    - results["mean_test_score"]
)

results = results.sort_values(
    by="mean_test_score",
    ascending=False
)

print("\n===== GRID SEARCH RESULTS =====")
print(results.to_string(index=False))

Fitting 3 folds for each of 36 candidates, totalling 108 fits

===== BEST MODEL =====
{'learning_rate': 0.1, 'n_estimators': 450}
Best CV PR-AUC: 0.368759

===== GRID SEARCH RESULTS =====
 param_learning_rate  param_n_estimators  mean_test_score  std_test_score  rank_test_score  mean_train_score  std_train_score      gap
                0.10                 450         0.368759        0.007329                1          0.381318         0.004211 0.012559
                0.08                 450         0.368664        0.007096                2          0.379286         0.003938 0.010622
                0.10                 400         0.368649        0.007270                3          0.380231         0.004156 0.011582
                0.10                 350         0.368586        0.007375                4          0.379336         0.004098 0.010751
                0.10                 300         0.368492        0.007496                5          0.378156         0.004016 0.009664
  

In [25]:
oof_proba = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=3,
    method="predict_proba",
    n_jobs=-1
)[:, 1]


oof_pred = (oof_proba >= 0.5).astype(int)

oof_roc_auc = roc_auc_score(y_train, oof_proba)
oof_pr_auc = average_precision_score(y_train, oof_proba)
oof_precision = precision_score(y_train, oof_pred)
oof_recall = recall_score(y_train, oof_pred)
oof_f1 = f1_score(y_train, oof_pred)


print("=== OOF EVALUATION ===")

print(f"ROC-AUC  : {oof_roc_auc:.6f}")
print(f"PR-AUC   : {oof_pr_auc:.6f}")
print(f"Precision: {oof_precision:.6f}")
print(f"Recall   : {oof_recall:.6f}")
print(f"F1       : {oof_f1:.6f}")


print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_train, oof_pred))


print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_train, oof_pred))

=== OOF EVALUATION ===
ROC-AUC  : 0.859235
PR-AUC   : 0.368536
Precision: 0.556687
Recall   : 0.103757
F1       : 0.174913

=== CONFUSION MATRIX ===
[[214928   1654]
 [ 17941   2077]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         0.0       0.92      0.99      0.96    216582
         1.0       0.56      0.10      0.17     20018

    accuracy                           0.92    236600
   macro avg       0.74      0.55      0.57    236600
weighted avg       0.89      0.92      0.89    236600



### Threshold Finetuning

In [26]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.25,0.30, 0.35, 0.40, 0.45, 0.50]

threshold_results = []

for threshold in thresholds:

    oof_pred_custom = (
        oof_proba >= threshold
    ).astype(int)

    precision = precision_score(
        y_train,
        oof_pred_custom
    )

    recall = recall_score(
        y_train,
        oof_pred_custom
    )

    f1 = f1_score(
        y_train,
        oof_pred_custom
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

    print(
        f"Threshold={threshold:.2f} | "
        f"Precision={precision:.3f} | "
        f"Recall={recall:.3f} | "
        f"F1={f1:.3f}"
    )


# ------------------------------------------------------------
# Select preferred threshold
# ------------------------------------------------------------

threshold_results = pd.DataFrame(
    threshold_results
)

best_threshold_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

pref_threshold = best_threshold_row["threshold"]


print("\n===== PREFERRED THRESHOLD =====")

print(
    f"Threshold: {pref_threshold:.2f}"
)

print(
    f"Precision: "
    f"{best_threshold_row['precision']:.3f}"
)

print(
    f"Recall   : "
    f"{best_threshold_row['recall']:.3f}"
)

print(
    f"F1       : "
    f"{best_threshold_row['f1']:.3f}"
)

Threshold=0.05 | Precision=0.190 | Recall=0.901 | F1=0.314
Threshold=0.10 | Precision=0.245 | Recall=0.786 | F1=0.373
Threshold=0.15 | Precision=0.291 | Recall=0.668 | F1=0.406
Threshold=0.20 | Precision=0.334 | Recall=0.552 | F1=0.416
Threshold=0.25 | Precision=0.374 | Recall=0.450 | F1=0.409
Threshold=0.30 | Precision=0.413 | Recall=0.360 | F1=0.385
Threshold=0.35 | Precision=0.453 | Recall=0.278 | F1=0.345
Threshold=0.40 | Precision=0.487 | Recall=0.208 | F1=0.291
Threshold=0.45 | Precision=0.524 | Recall=0.151 | F1=0.235
Threshold=0.50 | Precision=0.557 | Recall=0.104 | F1=0.175

===== PREFERRED THRESHOLD =====
Threshold: 0.20
Precision: 0.334
Recall   : 0.552
F1       : 0.416


### Test Set Evaluating

In [27]:
model_candidate = best_model

model_candidate.fit(X_train, y_train)
test_proba = model_candidate.predict_proba(X_test)[:, 1]
test_pred = (
    test_proba >= pref_threshold
).astype(int)




print("=== FINAL TEST EVALUATION ===")
print(f"Tested Model: {model_candidate}")

print("Threshold:", pref_threshold)

print("Accuracy :", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall   :", recall_score(y_test, test_pred))
print("F1       :", f1_score(y_test, test_pred))

print("ROC-AUC  :", roc_auc_score(y_test, test_proba))

print(
    "PR-AUC   :",
    average_precision_score(y_test, test_proba)
)



print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, test_pred))


print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_test,
        test_pred
    )
)

=== FINAL TEST EVALUATION ===
Tested Model: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=400, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=450, n_jobs=None,
              num_parallel_tree=None, ...)
Threshold: 0.2
Accuracy : 0.8697063447786174
Precision: 0.33568474823643885
Recall   : 0.5514485514485514
F1       : 0.41732819233386254
ROC-AUC  : 0.8615571589050701
PR-AUC   : 0.37712

# Random Search:

In [ ]:
param_distributions = {
    "learning_rate": [0.08, 0.10, 0.12, 0.13, 0.15, 0.17, 0.20],
    "max_depth": [3, 4, 5, 6, 7],
    "n_estimators": [200, 250, 275, 300, 325, 350, 400],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.05, 0.1, 0.2],
    "reg_alpha": [0, 0.05, 0.1, 0.2],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0]
}

model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search.fit(X_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)
best_model = random_search.best_estimator_

results_df = pd.DataFrame(random_search.cv_results_).drop(columns=["params", "split0_test_score", "split1_test_score", "split2_test_score", "mean_fit_time", "std_fit_time", "mean_score_time", "std_score_time"])
results_df = results_df.sort_values("rank_test_score")
print("\nRandom Search Results:")
print(results_df)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'subsample': 0.9, 'reg_lambda': 2.0, 'reg_alpha': 0, 'n_estimators': 325, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.17, 'gamma': 0.05, 'colsample_bytree': 0.8}
0.20153375340362167

Random Search Results:
    param_subsample  param_reg_lambda  param_reg_alpha  param_n_estimators  \
3               0.9               2.0             0.00                 325   
15              0.7               1.0             0.20                 275   
0               0.9               1.0             0.10                 250   
22              0.8               0.5             0.00                 250   
7               0.7               1.0             0.05                 275   
10              0.9               0.5             0.05                 400   
26              0.8               1.0             0.10                 275   
13              0.9               1.5             0.00                 275   
19              

In [29]:
print(best_model)
print("Model:", type(best_model).__name__)
print("Threshold:", pref_threshold)

print("\nFeatures:")
print(X_train.columns.tolist())

print("\nNumber of features:", len(X_train.columns))

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=400, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=450, n_jobs=None,
              num_parallel_tree=None, ...)
Model: XGBClassifier
Threshold: 0.2

Features:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentH

In [31]:
MODEL_DIR = Path("../model_detail")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "xgboost_model.pkl"

joblib.dump(best_model, model_path)

print(f"Model saved to: {model_path}")

Model saved to: ..\model_detail\xgboost_model.pkl


In [32]:
loaded_model = joblib.load(model_path)

print(type(loaded_model))

<class 'xgboost.sklearn.XGBClassifier'>


In [33]:
sample = X_test.iloc[:5]

original_pred = best_model.predict_proba(sample)[:, 1]
loaded_pred = loaded_model.predict_proba(sample)[:, 1]

print("Original model:")
print(original_pred)

print("\nLoaded model:")
print(loaded_pred)

Original model:
[0.17590734 0.33377796 0.02937165 0.09466712 0.03945534]

Loaded model:
[0.17590734 0.33377796 0.02937165 0.09466712 0.03945534]


In [35]:
metadata = {
    "model_name": "XGBoost Model 4",
    "target": "HeartDiseaseorAttack",
    "threshold": float(pref_threshold),

    "features": X_train.columns.tolist(),

    "hyperparameters": {
        "learning_rate": 0.10,
        "n_estimators": 450,
        "max_depth": 3,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "min_child_weight": 400,
        "gamma": 0.10,
        "reg_alpha": 0,
        "reg_lambda": 2
    },

    "test_metrics": {
        "accuracy": 0.8697063447786174,
        "precision": 0.33568474823643885,
        "recall": 0.5514485514485514,
        "f1": 0.41732819233386254,
        "roc_auc": 0.8615571589050701,
        "pr_auc": 0.37712175062559006
    }
}

metadata_path = MODEL_DIR / "model_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"Metadata saved to: {metadata_path}")

Metadata saved to: ..\model_detail\model_metadata.json
